# Combined normalized volume analysis - 5 min cadence, offset-aware

This version is intended for offset-corrected time columns.

Script extracts only 5-min-interval data:
- It detects 2.5-min acquisition cadence **within each source file and condition**.
- If a 2.5-min cadence is detected, it keeps the local 0, 5, 10, 15 min pattern and drops the local 2.5, 7.5, 12.5 min intermediate pattern.

This is important because after tile-scan offset correction, valid time series may start at e.g. 1.25 min or 1.75 min. Those should not be discarded.

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# User settings
# -----------------------------

INPUT_DIR = Path(r" *** \CombinedExperiments_Script4-5")
INPUT_GLOB = "*summary*_wide*.csv"

OUTPUT_DIR = INPUT_DIR / "combined_normalized_volume_output_5min_only_offset_aware"

# Nominal output bin spacing.
TIME_BIN_MIN = 5.0

# Drop intermediate points only when the file/condition was acquired at ~2.5-min cadence.
DROP_INTERMEDIATE_2P5_MIN_POINTS = True

# If the median interval is below this, the script treats the series as a 2.5-min-cadence acquisition.
# Your 2.5-min files have median intervals ~2.50 min; your 5-min files have ~5.00 min.
TWO_POINT_FIVE_CADENCE_THRESHOLD_MIN = 3.75

# When a 2.5-min cadence is detected, keep points close to the local 5-min cadence
# starting from the first available point, and drop the in-between points.
# 0.85 min is permissive enough for acquisition jitter, but far away from the 2.5-min intermediates.
LOCAL_5MIN_TOLERANCE_MIN = 0.85

# How to assign the remaining rows to output time bins.
#
# "nearest_global_grid":
#     corrected time 1.25 -> bin 0
#     corrected time 6.25 -> bin 5
#     This preserves the old output style, while mean_actual_time_min keeps the real offset information.
#
# "relative_to_first_kept_timepoint":
#     first kept point -> bin 0, second kept point -> bin 5, etc.
#     Use this only if you want every condition/file to start at nominal zero regardless of absolute offset.
TIME_BINNING_MODE = "nearest_global_grid"

# Plot x-axis:
# "time_min" = nominal binned time, usually 0, 5, 10, ...
# "mean_actual_time_min" = n-weighted actual corrected time of all rows contributing to the bin
PLOT_X_COLUMN = "time_min"

# Optional condition order for plotting and wide output.
# Conditions not listed here will be appended alphabetically.
PREFERRED_CONDITION_ORDER = ["mock", "Nivo", "Durva", "Enva", "mw11h317", "aPD", "aPDL", "aPD1", "aPDL1"]

# Plot settings
PLOT_SD_AS = "band"  # "band" or "errorbar"
FIGSIZE = (10, 6)
DPI = 300

## Helper functions

In [ ]:
def condition_sort_key(condition: str):
    """Sort conditions by preferred order, then alphabetically."""
    if condition in PREFERRED_CONDITION_ORDER:
        return (0, PREFERRED_CONDITION_ORDER.index(condition))
    return (1, condition.lower())


def detect_conditions(columns):
    """Detect conditions from '*_mean_normalized_volume' column names."""
    pattern = re.compile(r"^(?P<condition>.+)_mean_normalized_volume$")
    conditions = []
    for col in columns:
        match = pattern.match(col)
        if match:
            conditions.append(match.group("condition"))
    return sorted(set(conditions), key=condition_sort_key)


def assign_time_bins(part: pd.DataFrame) -> np.ndarray:
    """Assign kept rows to nominal output bins."""
    t = part["time_min_raw"].to_numpy(dtype=float)

    if TIME_BINNING_MODE == "nearest_global_grid":
        return np.floor(t / TIME_BIN_MIN + 0.5) * TIME_BIN_MIN

    if TIME_BINNING_MODE == "relative_to_first_kept_timepoint":
        return np.floor((t - t[0]) / TIME_BIN_MIN + 0.5) * TIME_BIN_MIN

    raise ValueError(
        "Unknown TIME_BINNING_MODE. Use 'nearest_global_grid' or "
        "'relative_to_first_kept_timepoint'."
    )


def drop_intermediate_2p5_rows(part: pd.DataFrame):
    """
    Detect 2.5-min cadence locally and drop the in-between points.

    Returns:
        kept, dropped, median_interval_min, detected_2p5_cadence
    """
    part = part.sort_values("time_min_raw").reset_index(drop=True).copy()
    part["row_in_condition_file"] = np.arange(len(part))

    if len(part) < 2 or not DROP_INTERMEDIATE_2P5_MIN_POINTS:
        return part, part.iloc[0:0].copy(), np.nan, False

    intervals = np.diff(part["time_min_raw"].to_numpy(dtype=float))
    median_interval = float(np.nanmedian(intervals))

    detected_2p5 = median_interval < TWO_POINT_FIVE_CADENCE_THRESHOLD_MIN

    if not detected_2p5:
        return part, part.iloc[0:0].copy(), median_interval, False

    # Local 5-min cadence starts from the first available row.
    # This keeps local 0, 5, 10, 15 ... and drops local 2.5, 7.5, 12.5 ...
    first_time = float(part["time_min_raw"].iloc[0])
    relative_time = part["time_min_raw"].to_numpy(dtype=float) - first_time
    local_bin = np.floor(relative_time / TIME_BIN_MIN + 0.5) * TIME_BIN_MIN
    local_deviation = np.abs(relative_time - local_bin)

    keep_mask = local_deviation <= LOCAL_5MIN_TOLERANCE_MIN

    # Safety fallback: if local-grid logic keeps suspiciously few rows,
    # use every-other-row logic instead.
    if keep_mask.sum() < max(2, len(part) // 3):
        keep_mask = (np.arange(len(part)) % 2) == 0
        method = "fallback every-other-row"
    else:
        method = "local 5-min grid"

    kept = part.loc[keep_mask].copy()
    dropped = part.loc[~keep_mask].copy()

    dropped["drop_reason"] = (
        f"2.5-min cadence detected; median interval = {median_interval:.3f} min. "
        f"Dropped local intermediate points using {method}."
    )

    return kept, dropped, median_interval, True

## Read files and convert to long format

In [ ]:
def read_one_summary_file(path: Path):
    """
    Read one wide summary file and return:
    - kept long rows
    - dropped intermediate rows
    - per-file/per-condition diagnostics
    """
    df = pd.read_csv(path)
    conditions = detect_conditions(df.columns)

    long_parts = []
    dropped_parts = []
    diagnostic_rows = []

    for condition in conditions:
        required = {
            "time_min": f"{condition}_time_min",
            "mean": f"{condition}_mean_normalized_volume",
            "sd": f"{condition}_sd_normalized_volume",
            "n": f"{condition}_n",
        }
        missing = [col for col in required.values() if col not in df.columns]
        if missing:
            warnings.warn(f"Skipping condition {condition!r} in {path.name}; missing columns: {missing}")
            continue

        part = pd.DataFrame({
            "source_file": path.name,
            "condition": condition,
            "time_min_raw": pd.to_numeric(df[required["time_min"]], errors="coerce"),
            "mean_normalized_volume": pd.to_numeric(df[required["mean"]], errors="coerce"),
            "sd_normalized_volume": pd.to_numeric(df[required["sd"]], errors="coerce"),
            "n": pd.to_numeric(df[required["n"]], errors="coerce"),
        })

        # Drop rows that do not contain a measurement.
        part = part.dropna(subset=["condition", "time_min_raw", "mean_normalized_volume", "n"])
        part = part[part["n"] > 0].copy()

        if part.empty:
            diagnostic_rows.append({
                "source_file": path.name,
                "condition": condition,
                "rows_input": 0,
                "rows_kept": 0,
                "rows_dropped_intermediate": 0,
                "median_interval_min": np.nan,
                "detected_2p5_cadence": False,
                "max_global_bin_deviation_kept_min": np.nan,
            })
            continue

        # SD is undefined/blank for n=1 in several files. For pooling, its within-group
        # contribution is zero because (n-1)*sd^2 = 0, so we can safely set it to zero.
        missing_sd_n1 = part["sd_normalized_volume"].isna() & (part["n"] <= 1)
        part.loc[missing_sd_n1, "sd_normalized_volume"] = 0.0

        # If n>1 but SD is missing, the pooled SD cannot be correctly reconstructed.
        problematic = part["sd_normalized_volume"].isna() & (part["n"] > 1)
        if problematic.any():
            bad = part.loc[problematic, ["source_file", "condition", "time_min_raw", "n"]].head(10)
            raise ValueError(
                "Found rows with n > 1 but missing SD. These rows cannot be pooled correctly.\\n"
                f"Examples:\\n{bad.to_string(index=False)}"
            )

        kept, dropped, median_interval, detected_2p5 = drop_intermediate_2p5_rows(part)

        if not kept.empty:
            kept["time_min"] = assign_time_bins(kept)
            kept["time_bin_deviation_min"] = (kept["time_min_raw"] - kept["time_min"]).abs()
            kept["median_interval_min"] = median_interval
            kept["detected_2p5_cadence"] = detected_2p5
            long_parts.append(kept)

        if not dropped.empty:
            dropped["time_min"] = assign_time_bins(dropped)
            dropped["time_bin_deviation_min"] = (dropped["time_min_raw"] - dropped["time_min"]).abs()
            dropped["median_interval_min"] = median_interval
            dropped["detected_2p5_cadence"] = detected_2p5
            dropped_parts.append(dropped)

        diagnostic_rows.append({
            "source_file": path.name,
            "condition": condition,
            "rows_input": len(part),
            "rows_kept": len(kept),
            "rows_dropped_intermediate": len(dropped),
            "median_interval_min": median_interval,
            "detected_2p5_cadence": detected_2p5,
            "max_global_bin_deviation_kept_min": kept["time_bin_deviation_min"].max() if not kept.empty else np.nan,
        })

    kept_out = pd.concat(long_parts, ignore_index=True) if long_parts else pd.DataFrame()
    dropped_out = pd.concat(dropped_parts, ignore_index=True) if dropped_parts else pd.DataFrame()
    diagnostics_out = pd.DataFrame(diagnostic_rows)

    return kept_out, dropped_out, diagnostics_out

## Pool across files

In [ ]:
def pooled_mean_sd(group: pd.DataFrame) -> pd.Series:
    """
    Pool summary statistics from multiple rows.

    Each row has mean_i, sd_i, n_i. The pooled sample SD reconstructs the
    variation of the underlying measurements as closely as possible from summary data.
    """
    n = group["n"].to_numpy(dtype=float)
    means = group["mean_normalized_volume"].to_numpy(dtype=float)
    sds = group["sd_normalized_volume"].to_numpy(dtype=float)
    times_raw = group["time_min_raw"].to_numpy(dtype=float)
    bin_deviation = group["time_bin_deviation_min"].to_numpy(dtype=float)

    total_n = np.nansum(n)
    if total_n <= 0:
        return pd.Series({
            "mean_normalized_volume": np.nan,
            "sd_normalized_volume": np.nan,
            "sem_normalized_volume": np.nan,
            "total_n": 0,
            "n_source_rows": len(group),
            "n_source_files": group["source_file"].nunique(),
            "mean_actual_time_min": np.nan,
            "mean_abs_time_bin_deviation_min": np.nan,
        })

    pooled_mean = np.nansum(n * means) / total_n

    if total_n > 1:
        within_ss = np.nansum((n - 1) * (sds ** 2))
        between_ss = np.nansum(n * ((means - pooled_mean) ** 2))
        pooled_var = (within_ss + between_ss) / (total_n - 1)
        pooled_sd = np.sqrt(max(pooled_var, 0.0))
        pooled_sem = pooled_sd / np.sqrt(total_n)
    else:
        pooled_sd = float(sds[0]) if not np.isnan(sds[0]) else np.nan
        pooled_sem = pooled_sd / np.sqrt(total_n) if total_n > 0 else np.nan

    # Diagnostics: n-weighted average of actual corrected timestamps and their bin deviation.
    mean_actual_time = np.nansum(n * times_raw) / total_n
    mean_abs_bin_deviation = np.nansum(n * bin_deviation) / total_n

    return pd.Series({
        "mean_normalized_volume": pooled_mean,
        "sd_normalized_volume": pooled_sd,
        "sem_normalized_volume": pooled_sem,
        "total_n": int(total_n) if float(total_n).is_integer() else total_n,
        "n_source_rows": len(group),
        "n_source_files": group["source_file"].nunique(),
        "mean_actual_time_min": mean_actual_time,
        "mean_abs_time_bin_deviation_min": mean_abs_bin_deviation,
    })


def combine_all_files(input_dir=INPUT_DIR, input_glob=INPUT_GLOB):
    """
    Load all files, convert to long form, remove 2.5-min intermediate points,
    and pool by condition and nominal time bin.
    """
    paths = sorted(Path(input_dir).glob(input_glob))
    if not paths:
        raise FileNotFoundError(f"No files matched {Path(input_dir) / input_glob}")

    raw_long_parts = []
    dropped_parts = []
    diagnostic_parts = []

    for path in paths:
        kept, dropped, diagnostics = read_one_summary_file(path)

        if not kept.empty:
            raw_long_parts.append(kept)
        if not dropped.empty:
            dropped_parts.append(dropped)
        if not diagnostics.empty:
            diagnostic_parts.append(diagnostics)

    if not raw_long_parts:
        raise ValueError("No valid condition blocks were found in the selected files.")

    raw_long = pd.concat(raw_long_parts, ignore_index=True)
    dropped_intermediate = (
        pd.concat(dropped_parts, ignore_index=True)
        if dropped_parts else
        pd.DataFrame()
    )
    file_condition_diagnostics = (
        pd.concat(diagnostic_parts, ignore_index=True)
        if diagnostic_parts else
        pd.DataFrame()
    )

    # Compatibility note:
    # This avoids the pandas >=2.2-only include_groups=False argument.
    combined_long = (
        raw_long
        .groupby(["condition", "time_min"], sort=False)
        .apply(pooled_mean_sd)
        .reset_index()
    )

    combined_long["condition"] = pd.Categorical(
        combined_long["condition"],
        categories=sorted(combined_long["condition"].unique(), key=condition_sort_key),
        ordered=True,
    )
    combined_long = combined_long.sort_values(["condition", "time_min"]).reset_index(drop=True)
    combined_long["condition"] = combined_long["condition"].astype(str)

    return raw_long, dropped_intermediate, file_condition_diagnostics, combined_long, paths

## Wide output and plotting

In [ ]:
def make_wide_output(combined_long: pd.DataFrame) -> pd.DataFrame:
    """
    Create a wide CSV similar to the input style:
    condition_time_min, condition_mean_normalized_volume, condition_sd_normalized_volume, condition_total_n, ...
    """
    conditions = sorted(combined_long["condition"].unique(), key=condition_sort_key)
    pieces = []

    for condition in conditions:
        sub = combined_long[combined_long["condition"] == condition].copy()
        sub = sub.sort_values("time_min")
        sub = sub[[
            "time_min",
            "mean_actual_time_min",
            "mean_abs_time_bin_deviation_min",
            "mean_normalized_volume",
            "sd_normalized_volume",
            "sem_normalized_volume",
            "total_n",
            "n_source_files",
            "n_source_rows",
        ]]
        sub.columns = [
            f"{condition}_time_min",
            f"{condition}_mean_actual_time_min",
            f"{condition}_mean_abs_time_bin_deviation_min",
            f"{condition}_mean_normalized_volume",
            f"{condition}_sd_normalized_volume",
            f"{condition}_sem_normalized_volume",
            f"{condition}_total_n",
            f"{condition}_n_source_files",
            f"{condition}_n_source_rows",
        ]
        sub = sub.reset_index(drop=True)
        pieces.append(sub)

    if not pieces:
        return pd.DataFrame()

    return pd.concat(pieces, axis=1)


def plot_combined(combined_long: pd.DataFrame, output_path: Path):
    """Plot mean normalized volume with pooled SD for each condition."""
    output_path = Path(output_path)
    conditions = sorted(combined_long["condition"].unique(), key=condition_sort_key)

    fig, ax = plt.subplots(figsize=FIGSIZE)

    for condition in conditions:
        sub = combined_long[combined_long["condition"] == condition].sort_values("time_min")
        x = sub[PLOT_X_COLUMN].to_numpy(dtype=float)
        y = sub["mean_normalized_volume"].to_numpy(dtype=float)
        sd = sub["sd_normalized_volume"].to_numpy(dtype=float)

        if PLOT_SD_AS.lower() == "errorbar":
            ax.errorbar(x, y, yerr=sd, marker="o", markersize=3, linewidth=1.5, capsize=2, label=condition)
        else:
            line, = ax.plot(x, y, marker="o", markersize=3, linewidth=1.5, label=condition)
            color = line.get_color()
            ax.fill_between(x, y - sd, y + sd, color=color, alpha=0.18, linewidth=0)

    x_label = "Time [min]"
    if PLOT_X_COLUMN == "time_min":
        x_label = "Nominal binned time [min]"
    elif PLOT_X_COLUMN == "mean_actual_time_min":
        x_label = "Mean actual corrected time [min]"

    ax.set_xlabel(x_label)
    ax.set_ylabel("Normalized IS volume")
    ax.set_title("Combined normalized interaction-site volume")
    ax.legend(frameon=False)
    ax.grid(True, alpha=0.25)

    fig.tight_layout()
    fig.savefig(output_path, dpi=DPI)
    plt.show()

## Run analysis

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_long, dropped_intermediate, file_condition_diagnostics, combined_long, input_paths = combine_all_files(
    INPUT_DIR,
    INPUT_GLOB,
)
combined_wide = make_wide_output(combined_long)

raw_long_path = OUTPUT_DIR / "all_input_summaries_long_diagnostic.csv"
dropped_timepoints_path = OUTPUT_DIR / "dropped_intermediate_timepoints.csv"
file_condition_diagnostics_path = OUTPUT_DIR / "file_condition_time_cadence_diagnostics.csv"
combined_long_path = OUTPUT_DIR / "combined_normalized_volume_long.csv"
combined_wide_path = OUTPUT_DIR / "combined_normalized_volume_wide.csv"
plot_path = OUTPUT_DIR / "combined_normalized_volume_plot.png"

raw_long.to_csv(raw_long_path, index=False)
dropped_intermediate.to_csv(dropped_timepoints_path, index=False)
file_condition_diagnostics.to_csv(file_condition_diagnostics_path, index=False)
combined_long.to_csv(combined_long_path, index=False)
combined_wide.to_csv(combined_wide_path, index=False)

plot_combined(combined_long, plot_path)

print(f"Loaded {len(input_paths)} files:")
for p in input_paths:
    print(f"  - {p.name}")

print()
print(f"Detected conditions: {', '.join(sorted(combined_long['condition'].unique(), key=condition_sort_key))}")
print(f"Kept rows after cadence filtering: {len(raw_long)}")
print(f"Dropped intermediate rows:         {len(dropped_intermediate)}")
print()
print(f"Raw diagnostic table:             {raw_long_path}")
print(f"Dropped time-point table:         {dropped_timepoints_path}")
print(f"File/cadence diagnostics table:   {file_condition_diagnostics_path}")
print(f"Combined long table:              {combined_long_path}")
print(f"Combined wide table:              {combined_wide_path}")
print(f"Plot:                             {plot_path}")

display(file_condition_diagnostics)
display(combined_long.head(20))